# PlantSeg × DINOv3: full experiment (Colab A100)

Upload this repository's `src/` folder to Google Drive as `MyDrive/plantseg-dinov3-lora/src` (it contains the code and `queue_initial.json`).
Select runtime type **A100 GPU** (high RAM) → *Runtime → Run all*. Keep the tab open.
When the queue finishes or stops, the last cell disconnects the runtime by itself (to save compute units).
If interrupted, do the same again: finished runs are skipped, an interrupted one resumes from its last checkpoint on Drive.

Monitoring (on Drive, under `ROOT`): `status.md` (human-readable), `log/<run>.log`, `results/<run>.json`, `queue.json`.

In [ ]:
# 1) Settings
ROOT = '/content/drive/MyDrive/plantseg-dinov3-lora'   # this folder holds src/ (copy of the repo's src/); outputs are written here
MEASURE_SPEED = False   # 40-iteration speed measurement for every unmeasured model family (~15 min total; for the budget estimate)
OPEN_TEST = False       # OFF by default. The test set is evaluated once, with the final selected settings only.
TEST_RUNS = 'h_v2L_lora_s0' # comma-separated run ids; if empty, all finished runs
AUTO_DISCONNECT = True  # disconnect the runtime when the queue finishes/stops
REQUIRE_A100 = True     # stop if not an A100 (time estimates are invalid on L4/T4)

In [ ]:
# 2) Drive + GPU + code
from google.colab import drive
drive.mount('/content/drive')
import os, shutil, subprocess
assert os.path.isdir(f'{ROOT}/src'), f'{ROOT}/src missing: upload the repo\'s src/ folder to Drive under {ROOT}'
gpu = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], capture_output=True, text=True).stdout.strip()
print('GPU:', gpu)
if REQUIRE_A100 and 'A100' not in gpu:
    raise SystemExit('Not an A100: set the runtime type to A100 (or REQUIRE_A100=False).')
shutil.copytree(f'{ROOT}/src', '/content/src', dirs_exist_ok=True)   # code runs from local disk, outputs go to Drive
os.environ['HF_HOME'] = '/content/hf'

In [ ]:
# 3) Dependencies (torch is Colab's own CUDA build; only these are pinned)
!pip -q install timm==1.0.30 imagehash==4.3.2
import torch, timm
print('torch', torch.__version__, '| timm', timm.__version__, '| bf16', torch.cuda.is_bf16_supported(), '| cpu', os.cpu_count())

In [ ]:
# 4) Data: PlantSeg v7 (Zenodo 17719108) -> /content (no data on Drive), size + md5 verification, preprocessing, pHash
!python /content/src/data.py --target /content/plantseg

In [ ]:
# 4b) Re-opens the deferred h_v2L_lora_s0 (DINOv2 ViT-L + LoRA) in the queue; idempotent. Skip if queue.json does not exist yet.
import json, os
p = f'{ROOT}/queue.json'
d = json.load(open(p))
for k in d['runs']:
    if k['id'] == 'h_v2L_lora_s0' and k['status'] == 'deferred':
        k['status'] = 'pending'
json.dump(d, open(p + '.tmp', 'w'), ensure_ascii=False, indent=1); os.replace(p + '.tmp', p)
print([(k['id'], k['status']) for k in d['runs']])

In [ ]:
# 5) Queue: runs in priority order (skips finished ones, resumes an interrupted one). status.md is updated every minute.
!python /content/src/runner.py --root "{ROOT}" --data /content/plantseg/prep {'--measure_speed' if MEASURE_SPEED else ''}

In [ ]:
# 6) TEST (off by default)
if OPEN_TEST:
    !python /content/src/evaluate_test.py --root "{ROOT}" --target /content/plantseg --runs "{TEST_RUNS}" --yes_test
else:
    print('test closed')

In [ ]:
# 7) Disconnect the runtime (saves compute units)
print(open(f'{ROOT}/status.md').read()[:3000])
if AUTO_DISCONNECT:
    from google.colab import runtime
    runtime.unassign()